# Evaluation Fundamentals: BLEU, ROUGE, and Why They Fail

Automatic evaluation metrics were the backbone of NLP research for over a decade. Understanding why they were adopted, where they break down, and what replaced them is essential context for modern LLM evaluation.

## The Pre-LLM Evaluation Problem

Before transformer models, NLP tasks had relatively constrained output spaces: machine translation produced one sentence, summarization produced a short paragraph. Researchers needed cheap, reproducible metrics that correlated with human judgments without requiring expensive annotation.

**BLEU** (Bilingual Evaluation Understudy, Papineni et al. 2002) solved this for translation by measuring n-gram overlap between a hypothesis and reference translations. It became the de facto standard for MT research for 20 years.

**ROUGE** (Recall-Oriented Understudy for Gisting Evaluation, Lin 2004) adapted the idea for summarization, emphasizing recall over precision since summaries should cover the reference content.

## BLEU: The Formula and Its Assumptions

BLEU computes a modified precision for n-grams (1 through 4), applies a brevity penalty to discourage short outputs, and takes the geometric mean. The key assumption: a good translation shares n-grams with human references.

This assumption holds reasonably well when outputs are constrained (translate this sentence one way). It breaks catastrophically when outputs have valid paraphrases the reference did not anticipate.

In [ ]:
from collections import Counter
from math import exp, log

def ngrams(tokens, n):
    return [tuple(tokens[i:i+n]) for i in range(len(tokens)-n+1)]

def modified_precision(hypothesis, references, n):
    hyp_ngrams = Counter(ngrams(hypothesis, n))
    if not hyp_ngrams:
        return 0.0
    clipped = {}
    for gram, count in hyp_ngrams.items():
        max_ref = max(Counter(ngrams(ref, n))[gram] for ref in references)
        clipped[gram] = min(count, max_ref)
    return sum(clipped.values()) / sum(hyp_ngrams.values())

def brevity_penalty(hypothesis, references):
    hyp_len = len(hypothesis)
    ref_len = min(len(r) for r in references)
    if hyp_len >= ref_len:
        return 1.0
    return exp(1 - ref_len/hyp_len)

def bleu(hypothesis: str, references: list, max_n: int = 4) -> float:
    hyp_tok = hypothesis.lower().split()
    ref_tok = [r.lower().split() for r in references]
    bp = brevity_penalty(hyp_tok, ref_tok)
    precisions = []
    for n in range(1, max_n+1):
        p = modified_precision(hyp_tok, ref_tok, n)
        precisions.append(log(p) if p > 0 else float('-inf'))
    if float('-inf') in precisions:
        return 0.0
    return bp * exp(sum(precisions)/max_n)

hyp = "the cat sat on the mat"
refs = ["the cat is on the mat", "there is a cat on the mat"]
print(f"BLEU: {bleu(hyp, refs):.4f}")

## ROUGE: Recall-Oriented Variants

ROUGE-N measures recall of reference n-grams in the hypothesis. ROUGE-L uses the Longest Common Subsequence (LCS), which is more flexible than exact n-gram matching and allows for reordering.

In [ ]:
def rouge_n(hypothesis: str, reference: str, n: int = 2) -> dict:
    hyp_tok = hypothesis.lower().split()
    ref_tok = reference.lower().split()
    hyp_ng = Counter(ngrams(hyp_tok, n))
    ref_ng = Counter(ngrams(ref_tok, n))
    overlap = sum((hyp_ng & ref_ng).values())
    precision = overlap / sum(hyp_ng.values()) if hyp_ng else 0.0
    recall = overlap / sum(ref_ng.values()) if ref_ng else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"precision": precision, "recall": recall, "f1": f1}

def lcs_length(a: list, b: list) -> int:
    m, n = len(a), len(b)
    dp = [[0]*(n+1) for _ in range(m+1)]
    for i in range(1, m+1):
        for j in range(1, n+1):
            if a[i-1] == b[j-1]:
                dp[i][j] = dp[i-1][j-1] + 1
            else:
                dp[i][j] = max(dp[i-1][j], dp[i][j-1])
    return dp[m][n]

def rouge_l(hypothesis: str, reference: str) -> dict:
    hyp_tok = hypothesis.lower().split()
    ref_tok = reference.lower().split()
    lcs = lcs_length(hyp_tok, ref_tok)
    precision = lcs / len(hyp_tok) if hyp_tok else 0.0
    recall = lcs / len(ref_tok) if ref_tok else 0.0
    f1 = 2*precision*recall/(precision+recall) if (precision+recall) > 0 else 0.0
    return {"lcs": lcs, "precision": precision, "recall": recall, "f1": f1}

summary = "the researchers found that the model performed well on most tasks"
reference = "the study showed the model achieved strong performance across tasks"
print("ROUGE-1:", rouge_n(summary, reference, 1))
print("ROUGE-2:", rouge_n(summary, reference, 2))
print("ROUGE-L:", rouge_l(summary, reference))

## Where These Metrics Break Down

The core problem: both metrics measure **surface-form overlap**, not semantic similarity or factual accuracy.

- **Paraphrase insensitivity**: 'The vehicle departed' scores near zero against 'The car left' despite identical meaning.
- **Factual hallucinations score well**: 'Paris is the capital of Germany' shares most n-grams with 'Paris is the capital of France'.
- **Length gaming**: systems learned to pad outputs to avoid the brevity penalty.
- **Multi-reference does not scale**: collecting enough references to cover valid paraphrases is expensive and never complete.

In [ ]:
cases = [
    ("The car left quickly", ["The vehicle departed rapidly"], "Paraphrase: same meaning"),
    ("Paris is the capital of Germany", ["Paris is the capital of France"], "Factual error: high overlap"),
    ("The model is good and works well in many situations", ["The model performs well"], "Length gaming"),
]

for hyp, refs, label in cases:
    score = bleu(hyp, refs)
    rl = rouge_l(hyp, refs[0])
    print(f"\n{label}")
    print(f"  Hyp: {hyp}")
    print(f"  Ref: {refs[0]}")
    print(f"  BLEU={score:.3f}  ROUGE-L F1={rl['f1']:.3f}")

## What Came After

The community moved through several stages after recognizing BLEU and ROUGE limitations:

**BERTScore** (Zhang et al. 2019): uses contextual embeddings (BERT) to measure semantic similarity rather than surface overlap. Substantially better correlation with human judgments on paraphrase-heavy tasks.

**MoverScore**: frames evaluation as earth-mover distance over token embeddings — penalizes semantic drift more gracefully than n-gram mismatch.

**Learned metrics (BLEURT, COMET)**: regression models trained directly on human ratings. Incorporate both semantic similarity and task-specific quality signals. COMET became the standard for MT evaluation by 2022.

**LLM-as-judge** (2023+): a capable LLM scores responses against a rubric. Enables nuanced, multi-dimensional evaluation that correlates with human preference at scale. Covered in the next notebook.

BLEU and ROUGE remain useful as cheap sanity checks and for tasks where surface overlap genuinely matters — structured output format compliance, code token overlap, slot-filling accuracy.